<a href="https://colab.research.google.com/github/haniyathul/TESIS-HANIY/blob/main/PEMBENTUKAN%20ESF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ================================================================
# ================================================================
# PEMBENTUKAN BASIS EIGENVECTOR SPATIAL FILTERING (ESF)
#
# INPUT:
# File matriks W yang sudah berada di ROOT GitHub:
#
# W_GRID10_KNN5.npz
# W_GRID10_DISTANCE.npz
# W_GRID10_IDW.npz
# ...
# W_GRID50_KNN5.npz
# W_GRID50_DISTANCE.npz
# W_GRID50_IDW.npz
#
# OUTPUT:
# 3 basis ESF per grid
#
# TOTAL:
# 5 grid × 3 W = 15 file basis ESF
# ================================================================
# ================================================================


# ================================================================
# 0. IMPORT
# ================================================================

import os
import gc
import zipfile

import numpy as np
import pandas as pd

import scipy.sparse as sp

from scipy.linalg import eigh
from pathlib import Path


# ================================================================
# 1. KONFIGURASI GITHUB
# ================================================================

REPO_URL = (
    "https://github.com/"
    "haniyathul/"
    "TESIS-HANIY.git"
)

REPO_DIR = Path(
    "/content/TESIS-HANIY"
)


# ================================================================
# 2. CLONE / UPDATE REPOSITORY
# ================================================================

if not REPO_DIR.exists():

    print(
        "Repository belum ada di Colab."
    )

    print(
        "Melakukan clone..."
    )

    os.system(
        f'git clone "{REPO_URL}" "{REPO_DIR}"'
    )

else:

    print(
        "Repository sudah ada."
    )

    print(
        "Melakukan git pull..."
    )

    os.system(
        f'git -C "{REPO_DIR}" pull'
    )


print(
    "\nRepository:"
)

print(
    REPO_DIR
)


# ================================================================
# 3. FOLDER OUTPUT ESF
# ================================================================

ESF_OUTPUT_DIR = Path(
    "/content/ESF_BASIS"
)

ESF_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "\nFolder output:"
)

print(
    ESF_OUTPUT_DIR
)


# ================================================================
# 4. METODE W
# ================================================================

W_METHODS = [

    "KNN5",

    "DISTANCE",

    "IDW"

]


# ================================================================
# 5. FUNGSI MENCARI FILE W
#
# CATATAN:
# File W kakak berada langsung di ROOT repository.
#
# Contoh:
# /content/TESIS-HANIY/W_GRID10_KNN5.npz
# ================================================================

def cari_file_w(
    grid_size,
    w_method
):

    filename = (

        f"W_GRID"
        f"{grid_size}_"
        f"{w_method}.npz"

    )


    W_FILE = (

        REPO_DIR
        /
        filename

    )


    if not W_FILE.exists():

        raise FileNotFoundError(

            f"""
============================================================
FILE W TIDAK DITEMUKAN
============================================================

File:
{filename}

Lokasi yang dicari:
{W_FILE}

Pastikan file sudah ada di ROOT repository TESIS-HANIY.
============================================================
"""
        )


    return W_FILE


# ================================================================
# 6. CEK FILE W YANG ADA
# ================================================================

print("\n")
print("=" * 100)

print(
    "CEK FILE W DI GITHUB"
)

print("=" * 100)


for grid_size in [

    10,
    20,
    30,
    40,
    50

]:

    for w_method in W_METHODS:

        filename = (

            REPO_DIR
            /
            (
                f"W_GRID"
                f"{grid_size}_"
                f"{w_method}.npz"
            )

        )


        status = (

            "✅ ADA"

            if filename.exists()

            else "❌ TIDAK ADA"

        )


        print(

            f"GRID {grid_size:02d}"
            f" | "
            f"{w_method:8s}"
            f" | "
            f"{status}"

        )


# ================================================================
# 7. FUNGSI MORAN'S I UNTUK SETIAP EIGENVECTOR
# ================================================================

def hitung_moran_eigenvector(
    eigenvectors,
    W
):

    n = W.shape[0]


    # ------------------------------------------------------------
    # Total bobot spasial
    # ------------------------------------------------------------

    S0 = float(
        W.sum()
    )


    # ------------------------------------------------------------
    # W × E
    # ------------------------------------------------------------

    W_E = (
        W
        @
        eigenvectors
    )


    # ------------------------------------------------------------
    # Pembilang Moran
    # ------------------------------------------------------------

    numerator = np.sum(

        eigenvectors
        *
        W_E,

        axis=0

    )


    # ------------------------------------------------------------
    # Penyebut Moran
    # ------------------------------------------------------------

    denominator = np.sum(

        eigenvectors ** 2,

        axis=0

    )


    # ------------------------------------------------------------
    # Moran I
    # ------------------------------------------------------------

    moran_I = (

        (n / S0)

        *

        (
            numerator
            /
            denominator
        )

    )


    return np.asarray(
        moran_I,
        dtype=float
    )


# ================================================================
# 8. FUNGSI MEMBENTUK SATU BASIS ESF
# ================================================================

def bentuk_satu_esf(
    grid_size,
    w_method
):


    # ============================================================
    # JUMLAH LOKASI
    # ============================================================

    n = (

        grid_size
        *
        grid_size

    )


    print("\n")
    print("=" * 110)

    print(

        f"PEMBENTUKAN BASIS ESF"
        f" | "
        f"GRID {grid_size}×{grid_size}"
        f" | "
        f"W = {w_method}"

    )

    print("=" * 110)


    # ============================================================
    # 9. CARI FILE W
    # ============================================================

    W_FILE = cari_file_w(

        grid_size=
            grid_size,

        w_method=
            w_method

    )


    print(
        "File W       :",
        W_FILE.name
    )


    # ============================================================
    # 10. LOAD W
    # ============================================================

    W = sp.load_npz(
        W_FILE
    ).tocsr()


    print(
        "Ukuran W     :",
        W.shape
    )


    # ------------------------------------------------------------
    # Validasi ukuran W
    # ------------------------------------------------------------

    assert W.shape == (

        n,
        n

    ), (

        f"""
Ukuran W salah.

Didapat:
{W.shape}

Seharusnya:
{(n, n)}
"""
    )


    # ============================================================
    # 11. CEK NILAI W
    # ============================================================

    if not np.isfinite(
        W.data
    ).all():

        raise ValueError(

            f"W GRID{grid_size} {w_method} "
            f"mengandung NaN/Inf."

        )


    print(
        "Jumlah nonzero:",
        W.nnz
    )


    print(
        "Total bobot   :",
        round(
            float(
                W.sum()
            ),
            6
        )
    )


    # ============================================================
    # 12. SIMETRISASI W UNTUK OPERATOR ESF
    #
    # W pada DGP tetap W asli.
    #
    # Untuk eigendecomposition digunakan:
    #
    # C = (W + W') / 2
    #
    # supaya operator ESF bersifat simetris dan menghasilkan
    # eigenvalue/eigenvector real.
    # ============================================================

    C = (

        W
        +
        W.T

    ) * 0.5


    C = C.tocsr()


    # ------------------------------------------------------------
    # Hilangkan diagonal
    # ------------------------------------------------------------

    C.setdiag(
        0
    )


    C.eliminate_zeros()


    # ============================================================
    # 13. PEMBENTUKAN OPERATOR ESF
    #
    # M = I - 11'/n
    #
    # Operator:
    #
    # Omega = M C M
    #
    # Dibentuk melalui double-centering.
    # ============================================================

    C_dense = C.toarray()


    row_mean = C_dense.mean(

        axis=1,

        keepdims=True

    )


    col_mean = C_dense.mean(

        axis=0,

        keepdims=True

    )


    grand_mean = float(

        C_dense.mean()

    )


    OMEGA = (

        C_dense

        -

        row_mean

        -

        col_mean

        +

        grand_mean

    )


    # ============================================================
    # 14. JAGA SIMETRI NUMERIK
    # ============================================================

    OMEGA = (

        OMEGA
        +
        OMEGA.T

    ) / 2.0


    print(
        "Operator MCM : ✅ SELESAI"
    )


    # ============================================================
    # 15. EIGENDECOMPOSITION
    # ============================================================

    print(
        "Eigendecomposition..."
    )


    eigenvalues, eigenvectors = eigh(

        OMEGA,

        overwrite_a=True,

        check_finite=False

    )


    print(
        "Eigendecomposition: ✅ SELESAI"
    )


    # ============================================================
    # 16. URUTKAN EIGENVALUE TERBESAR → TERKECIL
    # ============================================================

    order = np.argsort(

        eigenvalues

    )[::-1]


    eigenvalues = (

        eigenvalues[
            order
        ]

    )


    eigenvectors = (

        eigenvectors[
            :,
            order
        ]

    )


    # ============================================================
    # 17. HAPUS EIGENVALUE NUMERIK ~ 0
    #
    # Akibat centering terdapat komponen konstan ~0.
    # Komponen tersebut tidak digunakan.
    # ============================================================

    tolerance = (

        1e-10

        *

        max(

            1.0,

            np.max(

                np.abs(
                    eigenvalues
                )

            )

        )

    )


    keep = (

        np.abs(
            eigenvalues
        )

        >

        tolerance

    )


    eigenvalues = (

        eigenvalues[
            keep
        ]

    )


    eigenvectors = (

        eigenvectors[
            :,
            keep
        ]

    )


    # ============================================================
    # 18. ORIENTASI TANDA EIGENVECTOR
    #
    # Eigenvector E dan -E secara matematis sama.
    # Dibuat konsisten agar reproducible.
    # ============================================================

    for j in range(

        eigenvectors.shape[1]

    ):


        idx_max = np.argmax(

            np.abs(

                eigenvectors[
                    :,
                    j
                ]

            )

        )


        if (

            eigenvectors[
                idx_max,
                j
            ]

            <

            0

        ):

            eigenvectors[
                :,
                j
            ] *= -1


    # ============================================================
    # 19. HITUNG MORAN I SETIAP EIGENVECTOR
    #
    # Moran dihitung menggunakan W ASLI.
    # ============================================================

    moran_I = hitung_moran_eigenvector(

        eigenvectors=
            eigenvectors,

        W=
            W

    )


    # ============================================================
    # 20. IDENTIFIKASI EIGENVALUE POSITIF / NEGATIF
    # ============================================================

    positive_mask = (

        eigenvalues > 0

    )


    negative_mask = (

        eigenvalues < 0

    )


    positive_index = np.where(

        positive_mask

    )[0]


    negative_index = np.where(

        negative_mask

    )[0]


    n_positive = int(

        positive_mask.sum()

    )


    n_negative = int(

        negative_mask.sum()

    )


    # ============================================================
    # 21. RINGKASAN
    # ============================================================

    print("\n")

    print(
        "Jumlah lokasi       :",
        n
    )


    print(
        "Jumlah eigenvector  :",
        eigenvectors.shape[1]
    )


    print(
        "Eigenvalue positif  :",
        n_positive
    )


    print(
        "Eigenvalue negatif  :",
        n_negative
    )


    print(
        "Moran I minimum     :",
        round(
            float(
                moran_I.min()
            ),
            6
        )
    )


    print(
        "Moran I maksimum    :",
        round(
            float(
                moran_I.max()
            ),
            6
        )
    )


    # ============================================================
    # 22. FILE OUTPUT
    # ============================================================

    OUTPUT_FILE = (

        ESF_OUTPUT_DIR

        /

        (
            f"ESF_BASIS_GRID"
            f"{grid_size}_"
            f"{w_method}.npz"
        )

    )


    # ============================================================
    # 23. SAVE
    #
    # Disimpan compressed + float32 supaya lebih ringan.
    # ============================================================

    np.savez_compressed(

        OUTPUT_FILE,


        grid_size=
            np.array(
                grid_size,
                dtype=np.int32
            ),


        n=
            np.array(
                n,
                dtype=np.int32
            ),


        w_method=
            np.array(
                w_method
            ),


        eigenvalues=
            eigenvalues.astype(
                np.float32
            ),


        eigenvectors=
            eigenvectors.astype(
                np.float32
            ),


        moran_I=
            moran_I.astype(
                np.float32
            ),


        positive_mask=
            positive_mask.astype(
                np.int8
            ),


        positive_index=
            positive_index.astype(
                np.int32
            ),


        negative_index=
            negative_index.astype(
                np.int32
            )

    )


    # ============================================================
    # 24. UKURAN FILE
    # ============================================================

    size_mb = (

        OUTPUT_FILE
        .stat()
        .st_size

        /

        (
            1024 ** 2
        )

    )


    print("\n")

    print(
        "✅ FILE ESF BERHASIL"
    )


    print(
        "Nama file :",
        OUTPUT_FILE.name
    )


    print(
        "Ukuran    :",
        round(
            size_mb,
            2
        ),
        "MB"
    )


    # ============================================================
    # 25. HASIL SUMMARY
    # ============================================================

    hasil = {

        "Grid":
            f"{grid_size}x{grid_size}",

        "W_Method":
            w_method,

        "N":
            n,

        "Jumlah_Eigenvector":
            int(
                eigenvectors.shape[1]
            ),

        "Eigen_Positif":
            n_positive,

        "Eigen_Negatif":
            n_negative,

        "Moran_Min":
            float(
                moran_I.min()
            ),

        "Moran_Max":
            float(
                moran_I.max()
            ),

        "Ukuran_MB":
            size_mb,

        "File":
            OUTPUT_FILE.name

    }


    # ============================================================
    # 26. CLEAN MEMORY
    # ============================================================

    del W
    del C
    del C_dense
    del OMEGA
    del eigenvectors
    del eigenvalues
    del moran_I

    gc.collect()


    return hasil


# ================================================================
# 27. FUNGSI MEMBENTUK ESF PER GRID
#
# SATU GRID:
#
# KNN5
# DISTANCE
# IDW
#
# = 3 FILE
# ================================================================

def bentuk_esf_grid(
    grid_size
):

    print("\n")
    print("#" * 110)

    print(

        f"PEMBENTUKAN BASIS ESF "
        f"GRID {grid_size}×{grid_size}"

    )

    print("#" * 110)


    hasil_grid = []


    for w_method in W_METHODS:


        hasil = bentuk_satu_esf(

            grid_size=
                grid_size,

            w_method=
                w_method

        )


        hasil_grid.append(
            hasil
        )


    # ============================================================
    # SUMMARY GRID
    # ============================================================

    summary = pd.DataFrame(
        hasil_grid
    )


    print("\n")
    print("=" * 110)

    print(

        f"SUMMARY ESF "
        f"GRID {grid_size}×{grid_size}"

    )

    print("=" * 110)


    print(

        summary.to_string(
            index=False
        )

    )


    # ============================================================
    # SAVE SUMMARY PER GRID
    # ============================================================

    summary_file = (

        ESF_OUTPUT_DIR

        /

        (
            f"SUMMARY_ESF_GRID"
            f"{grid_size}.csv"
        )

    )


    summary.to_csv(

        summary_file,

        index=False

    )


    print(
        "\n✅ Summary:",
        summary_file.name
    )


    return summary


print("\n")
print("=" * 100)

print(
    "✅ FUNGSI PEMBENTUKAN BASIS ESF SIAP"
)

print("=" * 100)

Repository belum ada di Colab.
Melakukan clone...

Repository:
/content/TESIS-HANIY

Folder output:
/content/ESF_BASIS


CEK FILE W DI GITHUB
GRID 10 | KNN5     | ✅ ADA
GRID 10 | DISTANCE | ✅ ADA
GRID 10 | IDW      | ✅ ADA
GRID 20 | KNN5     | ✅ ADA
GRID 20 | DISTANCE | ✅ ADA
GRID 20 | IDW      | ✅ ADA
GRID 30 | KNN5     | ✅ ADA
GRID 30 | DISTANCE | ✅ ADA
GRID 30 | IDW      | ✅ ADA
GRID 40 | KNN5     | ✅ ADA
GRID 40 | DISTANCE | ✅ ADA
GRID 40 | IDW      | ✅ ADA
GRID 50 | KNN5     | ✅ ADA
GRID 50 | DISTANCE | ✅ ADA
GRID 50 | IDW      | ✅ ADA


✅ FUNGSI PEMBENTUKAN BASIS ESF SIAP


In [4]:
# ================================================================
# GRID 10 × 10
# ================================================================

ESF_GRID10 = bentuk_esf_grid(
    10
)



##############################################################################################################
PEMBENTUKAN BASIS ESF GRID 10×10
##############################################################################################################


PEMBENTUKAN BASIS ESF | GRID 10×10 | W = KNN5
File W       : W_GRID10_KNN5.npz
Ukuran W     : (100, 100)
Jumlah nonzero: 500
Total bobot   : 100.0
Operator MCM : ✅ SELESAI
Eigendecomposition...
Eigendecomposition: ✅ SELESAI


Jumlah lokasi       : 100
Jumlah eigenvector  : 99
Eigenvalue positif  : 36
Eigenvalue negatif  : 63
Moran I minimum     : -0.626146
Moran I maksimum    : 0.990637


✅ FILE ESF BERHASIL
Nama file : ESF_BASIS_GRID10_KNN5.npz
Ukuran    : 0.04 MB


PEMBENTUKAN BASIS ESF | GRID 10×10 | W = DISTANCE
File W       : W_GRID10_DISTANCE.npz
Ukuran W     : (100, 100)
Jumlah nonzero: 684
Total bobot   : 100.0
Operator MCM : ✅ SELESAI
Eigendecomposition...
Eigendecomposition: ✅ SELESAI


Jumlah lokasi       : 100
Jumlah ei

In [5]:
# ================================================================
# GRID 20 × 20
# ================================================================

ESF_GRID20 = bentuk_esf_grid(
    20
)



##############################################################################################################
PEMBENTUKAN BASIS ESF GRID 20×20
##############################################################################################################


PEMBENTUKAN BASIS ESF | GRID 20×20 | W = KNN5
File W       : W_GRID20_KNN5.npz
Ukuran W     : (400, 400)
Jumlah nonzero: 2000
Total bobot   : 400.0
Operator MCM : ✅ SELESAI
Eigendecomposition...
Eigendecomposition: ✅ SELESAI


Jumlah lokasi       : 400
Jumlah eigenvector  : 399
Eigenvalue positif  : 156
Eigenvalue negatif  : 243
Moran I minimum     : -0.636886
Moran I maksimum    : 1.016719


✅ FILE ESF BERHASIL
Nama file : ESF_BASIS_GRID20_KNN5.npz
Ukuran    : 0.57 MB


PEMBENTUKAN BASIS ESF | GRID 20×20 | W = DISTANCE
File W       : W_GRID20_DISTANCE.npz
Ukuran W     : (400, 400)
Jumlah nonzero: 2964
Total bobot   : 400.0
Operator MCM : ✅ SELESAI
Eigendecomposition...
Eigendecomposition: ✅ SELESAI


Jumlah lokasi       : 400
Juml

In [6]:
# ================================================================
# GRID 30 × 30
# ================================================================

ESF_GRID30 = bentuk_esf_grid(
    30
)



##############################################################################################################
PEMBENTUKAN BASIS ESF GRID 30×30
##############################################################################################################


PEMBENTUKAN BASIS ESF | GRID 30×30 | W = KNN5
File W       : W_GRID30_KNN5.npz
Ukuran W     : (900, 900)
Jumlah nonzero: 4500
Total bobot   : 900.0
Operator MCM : ✅ SELESAI
Eigendecomposition...
Eigendecomposition: ✅ SELESAI


Jumlah lokasi       : 900
Jumlah eigenvector  : 899
Eigenvalue positif  : 365
Eigenvalue negatif  : 534
Moran I minimum     : -0.642403
Moran I maksimum    : 1.01843


✅ FILE ESF BERHASIL
Nama file : ESF_BASIS_GRID30_KNN5.npz
Ukuran    : 2.88 MB


PEMBENTUKAN BASIS ESF | GRID 30×30 | W = DISTANCE
File W       : W_GRID30_DISTANCE.npz
Ukuran W     : (900, 900)
Jumlah nonzero: 6844
Total bobot   : 900.0
Operator MCM : ✅ SELESAI
Eigendecomposition...
Eigendecomposition: ✅ SELESAI


Jumlah lokasi       : 900
Jumla

In [7]:
# ================================================================
# GRID 40 × 40
# ================================================================

ESF_GRID40 = bentuk_esf_grid(
    40
)



##############################################################################################################
PEMBENTUKAN BASIS ESF GRID 40×40
##############################################################################################################


PEMBENTUKAN BASIS ESF | GRID 40×40 | W = KNN5
File W       : W_GRID40_KNN5.npz
Ukuran W     : (1600, 1600)
Jumlah nonzero: 8000
Total bobot   : 1600.0
Operator MCM : ✅ SELESAI
Eigendecomposition...
Eigendecomposition: ✅ SELESAI


Jumlah lokasi       : 1600
Jumlah eigenvector  : 1599
Eigenvalue positif  : 645
Eigenvalue negatif  : 954
Moran I minimum     : -0.637535
Moran I maksimum    : 1.016528


✅ FILE ESF BERHASIL
Nama file : ESF_BASIS_GRID40_KNN5.npz
Ukuran    : 9.1 MB


PEMBENTUKAN BASIS ESF | GRID 40×40 | W = DISTANCE
File W       : W_GRID40_DISTANCE.npz
Ukuran W     : (1600, 1600)
Jumlah nonzero: 12324
Total bobot   : 1600.0
Operator MCM : ✅ SELESAI
Eigendecomposition...
Eigendecomposition: ✅ SELESAI


Jumlah lokasi       : 

In [8]:
# ================================================================
# GRID 50 × 50
# ================================================================

ESF_GRID50 = bentuk_esf_grid(
    50
)



##############################################################################################################
PEMBENTUKAN BASIS ESF GRID 50×50
##############################################################################################################


PEMBENTUKAN BASIS ESF | GRID 50×50 | W = KNN5
File W       : W_GRID50_KNN5.npz
Ukuran W     : (2500, 2500)
Jumlah nonzero: 12500
Total bobot   : 2500.0
Operator MCM : ✅ SELESAI
Eigendecomposition...
Eigendecomposition: ✅ SELESAI


Jumlah lokasi       : 2500
Jumlah eigenvector  : 2499
Eigenvalue positif  : 1030
Eigenvalue negatif  : 1469
Moran I minimum     : -0.637002
Moran I maksimum    : 1.022442


✅ FILE ESF BERHASIL
Nama file : ESF_BASIS_GRID50_KNN5.npz
Ukuran    : 22.21 MB


PEMBENTUKAN BASIS ESF | GRID 50×50 | W = DISTANCE
File W       : W_GRID50_DISTANCE.npz
Ukuran W     : (2500, 2500)
Jumlah nonzero: 19404
Total bobot   : 2500.0
Operator MCM : ✅ SELESAI
Eigendecomposition...
Eigendecomposition: ✅ SELESAI


Jumlah lokasi    

In [9]:
# ================================================================
# CEK 15 FILE BASIS ESF
# ================================================================

EXPECTED_FILES = []


for grid_size in [

    10,
    20,
    30,
    40,
    50

]:

    for w_method in [

        "KNN5",
        "DISTANCE",
        "IDW"

    ]:

        EXPECTED_FILES.append(

            f"ESF_BASIS_GRID"
            f"{grid_size}_"
            f"{w_method}.npz"

        )


print("\n")
print("=" * 100)

print(
    "CEK FINAL 15 FILE BASIS ESF"
)

print("=" * 100)


JUMLAH_ADA = 0


for i, filename in enumerate(

    EXPECTED_FILES,

    start=1

):


    filepath = (

        ESF_OUTPUT_DIR
        /
        filename

    )


    if filepath.exists():


        JUMLAH_ADA += 1


        size_mb = (

            filepath.stat().st_size

            /

            (
                1024 ** 2
            )

        )


        print(

            f"{i:02d}. "
            f"✅ "
            f"{filename}"
            f" | "
            f"{size_mb:.2f} MB"

        )


    else:


        print(

            f"{i:02d}. "
            f"❌ "
            f"{filename}"

        )


print("\n")

print(
    "Jumlah file ditemukan:",
    JUMLAH_ADA
)


assert JUMLAH_ADA == 15, (

    f"""
FILE ESF BELUM LENGKAP.

Ditemukan:
{JUMLAH_ADA}

Seharusnya:
15
"""
)


print(
    "\n✅ SEMUA 15 BASIS ESF LENGKAP"
)



CEK FINAL 15 FILE BASIS ESF
01. ✅ ESF_BASIS_GRID10_KNN5.npz | 0.04 MB
02. ✅ ESF_BASIS_GRID10_DISTANCE.npz | 0.02 MB
03. ✅ ESF_BASIS_GRID10_IDW.npz | 0.02 MB
04. ✅ ESF_BASIS_GRID20_KNN5.npz | 0.57 MB
05. ✅ ESF_BASIS_GRID20_DISTANCE.npz | 0.32 MB
06. ✅ ESF_BASIS_GRID20_IDW.npz | 0.32 MB
07. ✅ ESF_BASIS_GRID30_KNN5.npz | 2.88 MB
08. ✅ ESF_BASIS_GRID30_DISTANCE.npz | 1.67 MB
09. ✅ ESF_BASIS_GRID30_IDW.npz | 1.67 MB
10. ✅ ESF_BASIS_GRID40_KNN5.npz | 9.10 MB
11. ✅ ESF_BASIS_GRID40_DISTANCE.npz | 5.48 MB
12. ✅ ESF_BASIS_GRID40_IDW.npz | 5.49 MB
13. ✅ ESF_BASIS_GRID50_KNN5.npz | 22.21 MB
14. ✅ ESF_BASIS_GRID50_DISTANCE.npz | 13.67 MB
15. ✅ ESF_BASIS_GRID50_IDW.npz | 13.70 MB


Jumlah file ditemukan: 15

✅ SEMUA 15 BASIS ESF LENGKAP


In [10]:
# ================================================================
# DOWNLOAD 15 FILE ESF SATU-SATU
# ================================================================

from google.colab import files


# ================================================================
# AMBIL 15 FILE ESF
# ================================================================

ESF_FILES = sorted(

    ESF_OUTPUT_DIR.glob(
        "ESF_BASIS_GRID*.npz"
    )

)


# ================================================================
# VALIDASI
# ================================================================

assert len(
    ESF_FILES
) == 15, (

    f"""
Jumlah file ESF = {len(ESF_FILES)}

Seharusnya = 15.
"""
)


print(
    "✅ 15 FILE ESF LENGKAP"
)

print(
    "Mulai download satu-satu..."
)


# ================================================================
# DOWNLOAD SATU-SATU
# ================================================================

for i, file in enumerate(
    ESF_FILES,
    start=1
):

    print(
        f"\n⬇️ DOWNLOAD {i}/15"
    )

    print(
        file.name
    )

    files.download(
        str(
            file
        )
    )


print("\n")
print("=" * 100)

print(
    "✅ SELESAI — 15 FILE ESF SUDAH DIPANGGIL UNTUK DOWNLOAD"
)

print("=" * 100)

✅ 15 FILE ESF LENGKAP
Mulai download satu-satu...

⬇️ DOWNLOAD 1/15
ESF_BASIS_GRID10_DISTANCE.npz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


⬇️ DOWNLOAD 2/15
ESF_BASIS_GRID10_IDW.npz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (36 outputs left out)


⬇️ DOWNLOAD 15/15
ESF_BASIS_GRID50_KNN5.npz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



✅ SELESAI — 15 FILE ESF SUDAH DIPANGGIL UNTUK DOWNLOAD
